In [ ]:
import json
import os
import pandas as pd
import numpy as np

DATA_DIR = '../input/competitions/cassava-leaf-disease-classification/'

with open(os.path.join(DATA_DIR, 'label_num_to_disease_map.json')) as f:
    disease_map = json.load(f)

df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))

# 숫자로 된 label을 실제 질병 이름으로 바꾼 열을 하나 더 추가
df['disease_name'] = df['label'].astype(str).map(disease_map)

In [9]:
from sklearn.model_selection import train_test_split

#클래스 불균형이 발생했으므로 label의 개수를 동일하게 split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"학습용 데이터 개수: {len(train_df)}장")
print(f"테스트용 데이터 개수: {len(test_df)}장")

학습용 데이터 개수: 17117장
테스트용 데이터 개수: 4280장


In [10]:
import torch
from torch.utils.data import Dataset

class CassavaDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform 
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        image_name = self.df.loc[index, 'image_id']
        label = self.df.loc[index, 'label']

        image_path = os.path.join(self.image_dir, image_name)
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) 
        
        if self.transform is not None:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

In [11]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

In [12]:
from torch.utils.data import DataLoader

train_dataset = CassavaDataset(df=train_df, image_dir=DATA_DIR+'train_images', transform=transform)
test_dataset = CassavaDataset(df=test_df, image_dir=DATA_DIR+'train_images', transform=transform)

train_loader = DataLoader(
    train_dataset, 
    batch_size=32,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=32, 
    shuffle=False,
    num_workers=2
)

In [13]:
from torchvision import models
import torch.nn as nn

# ResNet 모델 load
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 172MB/s]


In [14]:
# 출력층 개수 변경

n_features = model.fc.in_features
model.fc = nn.Linear(n_features, 5)

In [15]:
# 모델 GPU로 전송

if torch.cuda.is_available():
    print("yes")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

yes


In [16]:
#손실함수와 옵티마이저 설정

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-6)

In [17]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        outputs = model(images)
        
        loss = criterion(outputs, labels)

        #오차역전파
        loss.backward()

        #가중치 수정
        optimizer.step()

        running_loss += loss.item()
        
    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] 완료! - 평균 오답 점수(Loss): {epoch_loss:.4f}")
print("학습 완료")

Epoch [1/10] 완료! - 평균 오답 점수(Loss): 0.5966
Epoch [2/10] 완료! - 평균 오답 점수(Loss): 0.3264
Epoch [3/10] 완료! - 평균 오답 점수(Loss): 0.1436
Epoch [4/10] 완료! - 평균 오답 점수(Loss): 0.0702
Epoch [5/10] 완료! - 평균 오답 점수(Loss): 0.0506
Epoch [6/10] 완료! - 평균 오답 점수(Loss): 0.0570
Epoch [7/10] 완료! - 평균 오답 점수(Loss): 0.0375
Epoch [8/10] 완료! - 평균 오답 점수(Loss): 0.0367
Epoch [9/10] 완료! - 평균 오답 점수(Loss): 0.0304
Epoch [10/10] 완료! - 평균 오답 점수(Loss): 0.0405
학습 완료


In [18]:
from sklearn.metrics import accuracy_score, classification_report

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

final_accuracy = accuracy_score(all_labels, all_preds)
print(f"/n Test 정확도: {final_accuracy * 100:.2f}%")

print("\n Classification Report")

disease_names = ['CBB (0)', 'CBSD (1)', 'CGM (2)', 'CMD (3)', 'Healthy (4)']
print(classification_report(all_labels, all_preds, target_names=disease_names))

/n Test 정확도: 80.77%

 Classification Report
              precision    recall  f1-score   support

     CBB (0)       0.61      0.41      0.49       217
    CBSD (1)       0.64      0.60      0.62       438
     CGM (2)       0.68      0.67      0.67       477
     CMD (3)       0.89      0.96      0.92      2632
 Healthy (4)       0.64      0.53      0.58       516

    accuracy                           0.81      4280
   macro avg       0.69      0.63      0.66      4280
weighted avg       0.80      0.81      0.80      4280

